# Notebook 02 — Fine-tuning LoRA (Etapa 2)

O versionamento é uma **função** (`versionar_no_hub`) chamada automaticamente ao fim de cada config.
Você escolhe o tipo de versão (1/2/3) **uma vez, na Config 1**; a **Config 2 herda** essa escolha.
Cada config treina e, ao terminar, **cria a tag sozinha** no Hub.

**Fluxo:** #0 dataset → #1 ambiente + funções → #2 Config 1 (pergunta tipo + treina + versiona) →
#3 Config 2 (herda tipo + treina + versiona) → #4 recuperação (só se OOM) → comparação (Critério 3).

### 0. Verificar o dataset (produzido no notebook 01, no Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/multimodais/dados'
os.environ['DATA_DIR'] = DATA_DIR          # usado nas células de treino via "$DATA_DIR"

meta_path = os.path.join(DATA_DIR, 'metadata.jsonl')
assert os.path.exists(meta_path), f'metadata.jsonl não encontrado em {DATA_DIR}. Rode o notebook 01 primeiro.'
meta = [json.loads(l)['file_name'] for l in open(meta_path, encoding='utf-8')]
imgs = [f for f in os.listdir(DATA_DIR) if f.lower().endswith(('.jpg', '.png'))]
faltando = [m for m in meta if not os.path.exists(os.path.join(DATA_DIR, m))]
print(f'DATA_DIR = {DATA_DIR} | imagens = {len(imgs)} | metadata = {len(meta)} | faltando = {len(faltando)}')
assert imgs and not faltando, 'Dataset incompleto. Rode/complete o notebook 01 antes de treinar.'


### 1. Ambiente + função de versionamento (rode 1x)

Instala dependências, prepara o script de treino do `diffusers`, faz login no Hub e **define as funções**
`perguntar_tipo()` e `versionar_no_hub()` usadas pelas células de config.

In [ ]:
# torchao NÃO é necessário para LoRA e vinha quebrado (wheel py3.10 em ambiente py3.12) -> remove.
!pip uninstall -y diffusers torchao
!pip -q install "transformers" "accelerate" "datasets" "peft"
![ -d /content/diffusers ] || git clone --depth 1 https://github.com/huggingface/diffusers
!pip -q install -e /content/diffusers
%cd /content/diffusers/examples/text_to_image
!pip -q install -r requirements.txt

import os, re, torch
from google.colab import userdata
from huggingface_hub import login, HfApi

assert torch.cuda.is_available(), 'SEM GPU. Ambiente de execução -> Alterar tipo -> T4 GPU.'
assert os.environ.get('DATA_DIR'), 'Rode a célula #0 antes (ela define DATA_DIR).'
print('GPU:', torch.cuda.get_device_name(0), '| DATA_DIR:', os.environ['DATA_DIR'])

login(token=userdata.get('HF_TOKEN'))
REPO_ID = 'lamble-lambe/atelie'
api = HfApi(token=userdata.get('HF_TOKEN'))
SEMVER = re.compile(r'^v?(\d+)\.(\d+)\.(\d+)$')

def versao_producao():
    """Maior tag semver no Hub = versão em produção (0.0.0 se não houver)."""
    try:
        tags = [t.name for t in api.list_repo_refs(REPO_ID).tags]
    except Exception:
        tags = []
    vs = [tuple(map(int, m.groups())) for t in tags if (m := SEMVER.match(t))]
    return max(vs) if vs else (0, 0, 0)

def perguntar_tipo(nome):
    """Pergunta o tipo de versão ANTES do treino."""
    print(f'{nome}: que tipo de versão este treino representa?')
    print('  1 = BUG (patch)   2 = FEATURE (minor)   3 = BREAKING (major)')
    return int(input('Digite 1/2/3: ').strip())

def versionar_no_hub(tipo, descricao=''):
    """Lê a produção, incrementa o segmento certo e cria a tag no último push. Chamar APÓS o treino."""
    maj, mnr, pat = versao_producao()
    nova = {1: (maj, mnr, pat + 1), 2: (maj, mnr + 1, 0), 3: (maj + 1, 0, 0)}[tipo]
    tag = f'v{nova[0]}.{nova[1]}.{nova[2]}'
    rotulo = {1: 'BUG/patch', 2: 'FEATURE/minor', 3: 'BREAKING/major'}[tipo]
    api.create_tag(repo_id=REPO_ID, tag=tag, tag_message=descricao or rotulo)
    print(f'\n✅ Versionado: v{maj}.{mnr}.{pat} --({rotulo})--> {tag}')
    print(f'   Hub: https://huggingface.co/{REPO_ID}/tree/{tag}')
    print(f'   Carregar: pipe.load_lora_weights("{REPO_ID}", revision="{tag}")')
    return tag

print('\nAmbiente pronto. Funções disponíveis: perguntar_tipo(), versionar_no_hub().')


### 2. Config 1 — treina e versiona automaticamente (rank=8, lr=1e-4, 1500 passos)

Responda o tipo (para este dataset novo: **2 = feature**). Ao fim do treino, a tag é criada sozinha.
Sem `--validation_prompt` para evitar o OOM na validação final (a avaliação visual fica no notebook 03).

In [ ]:
tipo_cfg1 = perguntar_tipo('Config 1')   # <-- responda 1/2/3 ANTES; depois pode se ausentar

!accelerate launch \
  --num_processes=1 --num_machines=1 \
  --mixed_precision='fp16' --dynamo_backend='no' \
  train_text_to_image_lora.py \
  --pretrained_model_name_or_path='stable-diffusion-v1-5/stable-diffusion-v1-5' \
  --train_data_dir="$DATA_DIR" \
  --caption_column='text' \
  --mixed_precision='fp16' \
  --resolution=512 --random_flip \
  --train_batch_size=1 --gradient_accumulation_steps=4 \
  --max_train_steps=1500 --learning_rate=1e-4 --lr_scheduler='cosine' \
  --lr_warmup_steps=0 --rank=8 \
  --checkpointing_steps=500 --seed=42 \
  --output_dir='/content/atelie-lora-cfg1' \
  --push_to_hub --hub_model_id='lamble-lambe/atelie'

# Treino terminou e o push foi feito -> cria a tag automaticamente.
# (Se o treino acima morreu com OOM/SIGKILL, NÃO confie nesta tag: use a célula #4 de recuperação.)
versionar_no_hub(tipo_cfg1, 'Config 1: rank=8, lr=1e-4, 1500 passos, 35 imagens (estilo lambe-lambe)')


### 3. Config 2 — treina e versiona automaticamente (rank=4, lr=5e-5, 1000 passos)

Configuração mais conservadora (menos overfitting) para a comparação do Critério 3.
**Herda o tipo escolhido na Config 1** (mesma natureza de mudança) — não pergunta de novo se a Config 1 já rodou.

In [ ]:
# Herda o tipo escolhido na Config 1 (mesma natureza de mudança).
# Se a Config 1 não foi rodada nesta sessão, pergunta.
tipo_cfg2 = tipo_cfg1 if 'tipo_cfg1' in globals() else perguntar_tipo('Config 2')
print(f'Config 2 usando o mesmo tipo da Config 1: {tipo_cfg2}')

!accelerate launch \
  --num_processes=1 --num_machines=1 \
  --mixed_precision='fp16' --dynamo_backend='no' \
  train_text_to_image_lora.py \
  --pretrained_model_name_or_path='stable-diffusion-v1-5/stable-diffusion-v1-5' \
  --train_data_dir="$DATA_DIR" \
  --caption_column='text' \
  --mixed_precision='fp16' \
  --resolution=512 --random_flip \
  --train_batch_size=1 --gradient_accumulation_steps=4 \
  --max_train_steps=1000 --learning_rate=5e-5 --lr_scheduler='cosine' \
  --lr_warmup_steps=100 --rank=4 \
  --checkpointing_steps=500 --seed=42 \
  --output_dir='/content/atelie-lora-cfg2' \
  --push_to_hub --hub_model_id='lamble-lambe/atelie'

versionar_no_hub(tipo_cfg2, 'Config 2: rank=4, lr=5e-5, 1000 passos, warmup 100, 35 imagens')

### 4. Recuperação — SÓ se um treino morreu (OOM) e o push não completou

Os pesos são salvos em `<output_dir>/pytorch_lora_weights.safetensors` ANTES do push. Se o processo
morreu na validação/push, use esta célula: faz backup no Drive, sobe os pesos e versiona manualmente.

In [ ]:
import os, shutil

OUTPUT_DIR = '/content/atelie-lora-cfg1'   # ajuste: cfg1 ou cfg2 (a config que falhou)
TIPO       = 2                             # 1=bug / 2=feature / 3=breaking
DESCRICAO  = 'upload manual dos pesos após OOM'

pesos = os.path.join(OUTPUT_DIR, 'pytorch_lora_weights.safetensors')
assert os.path.exists(pesos), f'Não encontrei os pesos em {pesos}. Confira o OUTPUT_DIR.'

backup = '/content/drive/MyDrive/Colab Notebooks/multimodais/backup_lora'
os.makedirs(backup, exist_ok=True)
shutil.copy(pesos, os.path.join(backup, os.path.basename(OUTPUT_DIR) + '.safetensors'))
print('Backup no Drive:', backup)

api.upload_file(path_or_fileobj=pesos, path_in_repo='pytorch_lora_weights.safetensors',
                repo_id=REPO_ID, repo_type='model',
                commit_message=f'upload manual dos pesos ({os.path.basename(OUTPUT_DIR)})')
versionar_no_hub(TIPO, DESCRICAO)


## Comparação das configurações (Critério 3)

| Hiperparâmetro | Config 1 | Config 2 |
|---|---|---|
| `rank` | 8 | 4 |
| `learning_rate` | 1e-4 | 5e-5 |
| `max_train_steps` | 1500 | 1000 |
| `lr_warmup_steps` | 0 | 100 |
| tag no Hub | `v_._._` | `v_._._` |

Compare as duas versões via `revision=` no **notebook 03** (grade, CLIPScore, memorização) e justifique
no relatório qual você promoveu para produção (`LORA_REVISION` no Space) e por quê.